# Simulator Comparison

Climb the Braket simulator ladder: run a circuit on the free Local simulator, then compare it against the managed simulators (SV1, DM1, TN1) you would reach for at scale.

**Objectives:**
- Run a representative circuit on the Local simulator and read its results and timing
- Compare the four simulators (qubit limit, what each models, cost per minute, when to use each) from the GUIDE
- See how you *would* submit the same circuit to SV1 or DM1 (code shown, never executed here)
- Model on paper, in NumPy, the noise that only DM1 would show you
- Internalize the workflow discipline: develop on Local, validate at scale on SV1, study noise on DM1, QPU only when proven

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

# Deterministic sampling for reproducible printed output
np.random.seed(0)

# The only simulator we will actually run: free, instant, exact state vector.
device = LocalSimulator()
print("LocalSimulator ready -- exact state vector, free, no credentials.")

## 1. A representative circuit: the 3-qubit GHZ state

We need one circuit to carry through the whole ladder. The 3-qubit GHZ state
$(\ket{000} + \ket{111})/\sqrt{2}$ is ideal: it is genuinely entangled (so it exercises a
real simulator), its ideal outcome is unambiguous (only the correlated bitstrings `000` and
`111` can ever appear), and it is small enough to run instantly.

We build it the textbook way: a Hadamard on qubit 0 to make a superposition, then a chain of
CNOTs to spread that superposition across all three qubits. Note the big-endian convention used
here -- qubit 0 is the **leftmost** bit of each measured bitstring.

In [ ]:
# 3-qubit GHZ: H on qubit 0, then CNOT 0->1 and CNOT 1->2.
ghz = Circuit().h(0).cnot(0, 1).cnot(1, 2)

print(ghz)
print(f"\nQubit count: {ghz.qubit_count}")
print(f"Circuit depth: {ghz.depth}")

## 2. Run it on the Local simulator (real, free, instant)

This is the only cell in the notebook that actually executes a simulation. We time it and read
the measurement counts. Because the Local simulator computes the **exact** state vector, the
only bitstrings with nonzero amplitude are `000` and `111` -- the GHZ correlation -- so those
are the only outcomes shots can return.

In [ ]:
import time

SHOTS = 1000
t0 = time.perf_counter()
result = device.run(ghz, shots=SHOTS).result()
elapsed = time.perf_counter() - t0

counts = result.measurement_counts
print(f"Measurement counts: {dict(counts)}")
print(f"Total shots: {sum(counts.values())}")
print(f"Wall-clock time: {elapsed*1000:.1f} ms  (cost: $0.00)")

# --- Asserts: a GHZ state can ONLY yield the correlated bitstrings ---
assert set(counts).issubset({"000", "111"}), f"unexpected bitstrings: {set(counts)}"
assert counts.get("000", 0) + counts.get("111", 0) == SHOTS, "shots leaked to other outcomes"
print("\nOK: only the correlated bitstrings 000 and 111 appear.")

### Why the sampling is trustworthy: check the exact state vector

Raw shots fluctuate, so the *tight* check reads the *exact*
state vector the Local simulator computed and verify it analytically. The GHZ amplitude is
$1/\sqrt{2}$ on `000` (index 0) and on `111` (index 7), and exactly zero on the other six basis
states. Big-endian means index 7 = binary `111` = all three qubits excited.

In [ ]:
sv = ghz.state_vector()
print(f"State vector length: {len(sv)}  (= 2**{ghz.qubit_count})")

probs = np.abs(sv) ** 2
populated = np.flatnonzero(probs > 1e-9)
print(f"Populated basis indices: {populated.tolist()}  -> bitstrings "
      f"{[format(i, '03b') for i in populated]}")

# Analytic GHZ target: amplitude 1/sqrt(2) on index 0 (000) and index 7 (111).
target = np.zeros(8, dtype=complex)
target[0] = 1 / np.sqrt(2)
target[7] = 1 / np.sqrt(2)

assert np.array_equal(populated, np.array([0, 7])), "only 000 and 111 may be populated"
assert np.allclose(sv, target), "state vector does not match the ideal GHZ state"
print("\nOK: exact state vector equals (|000> + |111>) / sqrt(2).")

In [ ]:
# The SAMPLED probabilities (counts / shots) approximate the exact 1/2 each, up to
# shot noise -- they are not exactly 1/2 from finite samples. The exact equality was
# already proved from the state vector in the cell above; here we just confirm the
# sampling agrees and never leaks probability to a forbidden outcome.
m_probs = result.measurement_probabilities
print(f"sampled measurement_probabilities: {dict(m_probs)}")

assert set(m_probs).issubset({"000", "111"}), "sampling only ever yields supported outcomes"
assert np.isclose(m_probs.get("000", 0.0), 0.5, atol=0.05), "sampled P(000) approximates 1/2"
assert np.isclose(m_probs.get("111", 0.0), 0.5, atol=0.05), "sampled P(111) approximates 1/2"
print("OK: sampling approximates the exact P(000) = P(111) = 1/2, up to shot noise.")

## 3. The simulator ladder -- a comparison table

The Local run above cost nothing and finished in milliseconds. But a laptop holds only
$2^n$ amplitudes, so it tops out near 25 qubits, and it is noiseless. When you outgrow it you
climb a ladder of **managed** simulators, each a defense against wasting money on a real QPU.

The figures below are hardcoded from the GUIDE -- they are the facts you reason with, not
something we query (querying devices needs AWS credentials). Each rung answers a different need.

In [ ]:
# Hardcoded from GUIDE.md -- the simulator ladder.
LADDER = [
    {
        "name": "Local",
        "kind": "exact state vector",
        "max_qubits": 25,          # ~25; each qubit doubles memory (2**n amplitudes)
        "models_noise": False,
        "cost_per_min": 0.0,       # free
        "use_when": "develop and debug (default)",
    },
    {
        "name": "SV1",
        "kind": "state vector (managed)",
        "max_qubits": 34,
        "models_noise": False,
        "cost_per_min": 0.075,
        "use_when": "validate a gate circuit at scale, noiselessly",
    },
    {
        "name": "DM1",
        "kind": "density matrix",
        "max_qubits": 17,
        "models_noise": True,      # the ONLY simulator that models noise
        "cost_per_min": 0.075,
        "use_when": "study how noise degrades results",
    },
    {
        "name": "TN1",
        "kind": "tensor network",
        "max_qubits": 50,          # ~50, for large but lightly entangled circuits
        "models_noise": False,
        "cost_per_min": 0.275,
        "use_when": "large but lightly entangled circuits",
    },
]

header = f"{'Simulator':<10}{'Models':<22}{'Max qubits':<12}{'Noise?':<8}{'$/min':<8}When to use"
print(header)
print("-" * len(header))
for row in LADDER:
    cost = "free" if row["cost_per_min"] == 0 else f"${row['cost_per_min']:.3f}"
    noise = "yes" if row["models_noise"] else "no"
    print(f"{row['name']:<10}{row['kind']:<22}{row['max_qubits']:<12}"
          f"{noise:<8}{cost:<8}{row['use_when']}")

# --- Asserts: the table matches the GUIDE's invariants ---
by_name = {r["name"]: r for r in LADDER}
assert set(by_name) == {"Local", "SV1", "DM1", "TN1"}, "expected exactly four rungs"
assert by_name["Local"]["cost_per_min"] == 0.0, "Local is free"
assert by_name["SV1"]["max_qubits"] == 34 and by_name["SV1"]["cost_per_min"] == 0.075
assert by_name["DM1"]["max_qubits"] == 17 and by_name["DM1"]["cost_per_min"] == 0.075
assert by_name["TN1"]["cost_per_min"] == 0.275
# DM1 is the one and only noise modeler on the ladder.
noise_sims = [r["name"] for r in LADDER if r["models_noise"]]
assert noise_sims == ["DM1"], f"only DM1 models noise, got {noise_sims}"
print("\nOK: ladder matches the GUIDE; DM1 is the sole noise-modeling simulator.")

Visualize the two axes that decide which rung you reach for: scale (qubit ceiling) and price.
Local is free but small; SV1 buys you reach; TN1 buys you even more (for the right circuits);
DM1 trades qubits for the one thing the others cannot give you -- noise.

In [ ]:
names = [r["name"] for r in LADDER]
qubits = [r["max_qubits"] for r in LADDER]
costs = [r["cost_per_min"] for r in LADDER]
colors = ["#2a9d8f" if r["models_noise"] else "#264653" for r in LADDER]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(names, qubits, color=colors)
ax1.set_title("Max qubits per simulator")
ax1.set_ylabel("qubits")
for i, q in enumerate(qubits):
    ax1.text(i, q + 0.5, str(q), ha="center")

ax2.bar(names, costs, color=colors)
ax2.set_title("Cost per minute (USD)")
ax2.set_ylabel("$ / min")
for i, c in enumerate(costs):
    label = "free" if c == 0 else f"${c:.3f}"
    ax2.text(i, c + 0.006, label, ha="center")

fig.suptitle("The simulator ladder (teal = models noise)")
plt.tight_layout()
plt.show()

## 4. How you would submit the same circuit to SV1 or DM1

The Local run is enough for development. To validate at scale or study noise you would point the
same circuit at a managed simulator. That requires AWS credentials and **costs money per
minute**, so we do *not* execute it here -- the code below is shown as text only. Note that DM1
is where you would attach a noise channel; SV1 and TN1 cannot model noise at all.

In [ ]:
# Shown as a string and printed -- NOT executed. Running it would require AWS
# credentials and incur per-minute charges. Develop locally first; reach for these
# only once the algorithm is proven.

sv1_submission = '''
from braket.aws import AwsDevice

sv1 = AwsDevice("arn:aws:braket:::device/quantum-simulator/amazon/sv1")
# Same `ghz` circuit, validated at a scale a laptop cannot hold (up to 34 qubits).
task = sv1.run(ghz, shots=1000)          # billed at $0.075 / minute
counts = task.result().measurement_counts
'''

dm1_submission = '''
from braket.aws import AwsDevice
from braket.circuits import Noise

dm1 = AwsDevice("arn:aws:braket:::device/quantum-simulator/amazon/dm1")
# DM1 is the ONLY simulator that models noise: attach a channel, then submit.
noisy = ghz.copy()
noisy.apply_gate_noise(Noise.Depolarizing(probability=0.05))
task = dm1.run(noisy, shots=1000)        # billed at $0.075 / minute
counts = task.result().measurement_counts  # now smeared away from clean 000 / 111
'''

print("# --- SV1 submission (NOT run here) ---")
print(sv1_submission)
print("# --- DM1 submission with noise (NOT run here) ---")
print(dm1_submission)

## 5. What DM1 would show -- modeled here in NumPy

We cannot run DM1 in the browser, but we can build, by hand in NumPy, the density matrix it
would compute. A pure state has density matrix $\rho = \ket{\psi}\bra{\psi}$. A depolarizing
channel of strength $p$ mixes in the maximally mixed state:
$\rho' = (1-p)\,\rho + p\,\tfrac{I}{2^n}$. The diagonal still concentrates on `000` and `111`,
but the **off-diagonal coherence** -- the quantum correlation between them -- shrinks by
$(1-p)$. That decay is exactly what DM1 exists to reveal, and why a noiseless SV1 cannot.

In [ ]:
dim = 2 ** ghz.qubit_count           # 8
psi = ghz.state_vector()
rho_ideal = np.outer(psi, psi.conj())  # pure-state density matrix

p = 0.05                              # depolarizing strength
rho_noisy = (1 - p) * rho_ideal + p * np.eye(dim) / dim

# The 000<->111 coherence lives at indices (0, 7). Ideally it is 1/2.
coh_ideal = rho_ideal[0, 7].real
coh_noisy = rho_noisy[0, 7].real
print(f"Coherence rho[000,111]:  ideal = {coh_ideal:.4f}   noisy = {coh_noisy:.4f}")
print(f"Diagonal P(000), P(111):  ideal = {rho_ideal[0,0].real:.4f}, {rho_ideal[7,7].real:.4f}"
      f"   noisy = {rho_noisy[0,0].real:.4f}, {rho_noisy[7,7].real:.4f}")

# --- Asserts: a valid density matrix, with coherence damped by exactly (1-p) ---
assert np.isclose(np.trace(rho_ideal).real, 1.0), "rho must have unit trace"
assert np.isclose(np.trace(rho_noisy).real, 1.0), "noisy rho must still have unit trace"
assert np.isclose(coh_noisy, (1 - p) * coh_ideal), "coherence should scale by (1 - p)"
assert coh_noisy < coh_ideal, "noise must reduce coherence"
print("\nOK: trace preserved, coherence damped by (1 - p) -- the decay DM1 would report.")

In [ ]:
# Side by side: the ideal density matrix vs the depolarized one DM1 would return.
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, mat, title in [
    (axes[0], rho_ideal.real, "Ideal rho (SV1 / Local)"),
    (axes[1], rho_noisy.real, "Depolarized rho (DM1, p=0.05)"),
]:
    im = ax.imshow(mat, cmap="viridis", vmin=0, vmax=0.5)
    ax.set_title(title)
    ax.set_xticks(range(dim))
    ax.set_yticks(range(dim))
    ax.set_xticklabels([format(i, "03b") for i in range(dim)], rotation=90, fontsize=7)
    ax.set_yticklabels([format(i, "03b") for i in range(dim)], fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("The off-diagonal corners (000<->111) fade under noise")
plt.tight_layout()
plt.show()

## 6. The workflow discipline

The ladder is not a menu -- it is an order. Climb it rung by rung, and only step onto a QPU once
the algorithm is proven. Skipping rungs is how you burn a budget.

1. **Develop on Local** -- free, instant, no queue. Everything above started here.
2. **Validate at scale on SV1** -- when the circuit outgrows your laptop's $2^n$ memory.
3. **Study noise on DM1** -- the only simulator that shows you the decay a real machine will.
4. **QPU only when proven** -- per task plus per shot; always estimate cost first.

In [ ]:
WORKFLOW = [
    ("Local", "develop & debug", 0.0),
    ("SV1",   "validate at scale", 0.075),
    ("DM1",   "study noise",       0.075),
    ("QPU",   "only when proven",  None),  # metered per task + per shot
]
for i, (stage, why, cost) in enumerate(WORKFLOW, 1):
    price = "free" if cost == 0 else ("per task + per shot" if cost is None else f"${cost}/min")
    print(f"  {i}. {stage:<6} -> {why:<20} ({price})")

# The discipline is ordered: Local must come first, QPU last.
stages = [w[0] for w in WORKFLOW]
assert stages[0] == "Local" and stages[-1] == "QPU", "develop on Local first, QPU last"
print("\nOK: develop on Local -> validate on SV1 -> study noise on DM1 -> QPU last.")

## Exercises

Work these on the Local simulator only -- no credentials, no cost.

In [ ]:
# Exercise 1: Scale the GHZ state.
# Build a 5-qubit GHZ with H(0) followed by a chain of CNOTs, run it on the
# Local simulator, and assert only '00000' and '11111' appear in the counts.
#
# TODO: your code here
# ghz5 = Circuit().h(0)
# for q in range(4):
#     ghz5 = ghz5.cnot(q, q + 1)
# counts5 = device.run(ghz5, shots=1000).result().measurement_counts
# assert set(counts5).issubset({'00000', '11111'})

In [ ]:
# Exercise 2: Estimate the SV1 bill (do NOT run on SV1).
# SV1 costs $0.075/min. If a validation sweep of 40 circuits averages 3 seconds
# of compute each, what does it cost? Compute it in pure Python.
#
# TODO: your code here
# n_circuits = 40
# seconds_each = 3
# minutes = n_circuits * seconds_each / 60
# cost = minutes * 0.075
# print(f'Estimated SV1 cost: ${cost:.4f}')

In [ ]:
# Exercise 3: Push the noise model.
# Reuse rho_ideal from section 5. Sweep the depolarizing strength p over
# [0.0, 0.1, 0.2, 0.4] and record the 000<->111 coherence rho'[0, 7].
# Confirm it falls as (1 - p) * 0.5. Plot coherence vs p.
#
# TODO: your code here
# ps = [0.0, 0.1, 0.2, 0.4]
# cohs = []
# for p in ps:
#     rho_p = (1 - p) * rho_ideal + p * np.eye(dim) / dim
#     cohs.append(rho_p[0, 7].real)
# plt.plot(ps, cohs, 'o-')
# plt.xlabel('depolarizing p'); plt.ylabel('coherence rho[000,111]')
# plt.show()

## Summary

- The **Local** simulator runs the exact state vector for free and instantly -- the only thing we actually executed. Our 3-qubit GHZ returned only the correlated bitstrings `000` and `111`, and its exact state vector equals $(\ket{000}+\ket{111})/\sqrt{2}$.
- The **ladder**: Local (free, ~25 qubits, exact) -> SV1 (\$0.075/min, 34 qubits, exact) -> DM1 (\$0.075/min, 17 qubits, the only one that models **noise**) -> TN1 (\$0.275/min, ~50 qubits, lightly entangled).
- Submitting to SV1 or DM1 needs AWS credentials and per-minute charges, so that code is shown as text, never run. DM1 is where you attach a noise channel.
- We modeled DM1's job in NumPy: depolarizing noise damps the `000<->111` coherence by $(1-p)$ while preserving trace -- the decay a noiseless simulator can never show you.
- The discipline is ordered: **develop on Local -> validate at scale on SV1 -> study noise on DM1 -> QPU only when proven.** Skipping rungs burns budget.

**Next:** [06-noise-and-errors.ipynb](06-noise-and-errors.ipynb) -- add depolarizing and amplitude-damping channels, compare noisy vs ideal results, and meet zero-noise extrapolation.